# Stage 12: Cross-Species Transfer and Exercise-Axis Preservation

> **References shown: refs4** (the jointly re-annotated references, `references_v4/`; default since 2026-10-31, commit 904dfb2), **model: retrain v2** (`checkpoint-591763`). Outputs are read from the refs4 folders. The July-reference version of this notebook is in git history at commit `1281d21`; the July-vs-refs4 comparison is `deconvolution/REFS4_RERUN_RESULTS.md`, and April-vs-v2 is `docs/RETRAIN_V2_RESULTS.md`. The transfer comparison is by the gate method (10 stratified 5-fold splits, 10,000 permutations; `genecompass_input_v2_refs4_human/gate_rcv10/`).

## Stage Objective

Project rat pseudo-cell expression into the human GeneCompass token space and test whether rat exercise-associated latent axes remain detectable.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Ortholog projection | Map rat expression features to eligible human ortholog tokens and quantify mapped expression mass. |
| 2 | Human-space embedding | Embed projected pseudo-cells with the same model using the human species identifier. |
| 3 | Transfer validity | Evaluate preservation of the sex axis as a pre-specified positive-control gate. |
| 4 | Exercise-axis comparison | Compare rat- and human-space supervised-probe metrics for all blocks and Stage 11 hotspots. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 8 pseudo-cells, Stage 3 ortholog maps, Stage 7 model, and Stage 11 hotspot definitions. |
| Primary outputs | Human-space embeddings, transfer-comparison statistics, and hotspot preservation classifications. |

## Result Overview

All 11 refs4 rat-space exercise hotspots keep their exercise signal in human space (11 preserved, 0 weakened, 0 lost). The sex positive control passes (median sex AUC rat 0.717 → human 0.730; block by block Spearman 0.918). Transferred cells also land on their Tabula Sapiens counterparts four to five times as often as chance (`translation/ATLAS_PROJECTION_RESULTS.md`), except for three hotspots: SKMGN tenocytes, LUNG vascular smooth muscle and BLOOD erythroid cells.

The sections below present the computational evidence in execution order. All reported figures are generated from the recorded stage outputs rather than manually entered plotting values unless explicitly identified as configuration constants.


In [1]:
import pathlib, os
GCIH = pathlib.Path(
    "../data/deconvolution/genecompass_input_v2_refs4_human"
)
GATE = GCIH / "gate_rcv10"   # transfer comparison by the gate method
tissues = ["bat","blood","cortex","heart","hippoc","hypoth","kidney","liver","lung","skmgn","skmvl","smlint","testes","watsc"]
print("=== human-space embedding files ===")
total_cells = 0
for t in tissues:
    emb = GCIH / t / "embeddings" / "cell_embeddings.npy"
    if emb.exists():
        import numpy as np
        arr = np.load(str(emb), mmap_mode="r")
        total_cells += arr.shape[0]
        print(f"  {t:<8}: {arr.shape[0]:5d} cells × {arr.shape[1]} dims  "
              f"({emb.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  {t:<8}: MISSING")

print(f"  TOTAL: {total_cells} human-space embeddings")

print("\n=== comparison outputs ===")
for fn in ["subspace_probe.tsv","transfer_comparison.tsv","transfer_comparison.md"]:
    p = GATE / fn
    sz = f"{p.stat().st_size/1024:.1f} KB" if p.exists() else "MISSING"
    print(f"  {fn}: {sz}")


=== human-space embedding files ===
  bat     :   300 cells × 768 dims  (0.9 MB)
  blood   :   650 cells × 768 dims  (2.0 MB)
  cortex  :   600 cells × 768 dims  (1.8 MB)
  heart   :   800 cells × 768 dims  (2.5 MB)
  hippoc  :   550 cells × 768 dims  (1.7 MB)
  hypoth  :   650 cells × 768 dims  (2.0 MB)
  kidney  :   750 cells × 768 dims  (2.3 MB)
  liver   :   400 cells × 768 dims  (1.2 MB)
  lung    :  1012 cells × 768 dims  (3.1 MB)
  skmgn   :   450 cells × 768 dims  (1.4 MB)
  skmvl   :   450 cells × 768 dims  (1.4 MB)
  smlint  :   650 cells × 768 dims  (2.0 MB)
  testes  :   150 cells × 768 dims  (0.5 MB)
  watsc   :   650 cells × 768 dims  (2.0 MB)
  TOTAL: 8062 human-space embeddings

=== comparison outputs ===
  subspace_probe.tsv: 36.7 KB
  transfer_comparison.tsv: 21.8 KB
  transfer_comparison.md: 2.3 KB


## Steps 1-2: Ortholog Projection and Human-Space Embedding

The Stage 3 map contains 15,234 rat-to-human ortholog pairs. Eligible human features must also occur in the human token vocabulary and median dictionary.

| Tissue | Rat genes | Orthologs (%) | Eligible | Count mass (%) | Pseudo-cells | Embedded |
|--------|----------:|--------------:|---------:|---------------:|-------------:|---------:|
| bat | 14,162 | 82.6 | 11,565 | 83.8 | 300 | 300 |
| blood | 14,920 | 81.4 | 11,959 | 78.8 | 650 | 650 |
| cortex | 15,385 | 82.0 | 12,460 | 86.6 | 600 | 600 |
| heart | 15,864 | 81.3 | 12,734 | 84.7 | 800 | 800 |
| hippoc | 15,313 | 81.5 | 12,316 | 86.7 | 550 | 550 |
| hypoth | 14,061 | 83.1 | 11,597 | 85.9 | 650 | 650 |
| kidney | 14,892 | 81.8 | 12,034 | 84.4 | 750 | 750 |
| liver | 12,766 | 83.0 | 10,474 | 81.5 | 400 | 400 |
| lung | 15,233 | 82.4 | 12,416 | 84.5 | 1,050 | 1,012 |
| skmgn | 13,142 | 83.3 | 10,842 | 86.2 | 450 | 450 |
| skmvl | 13,141 | 83.3 | 10,841 | 85.4 | 450 | 450 |
| smlint | 12,874 | 82.6 | 10,475 | 80.5 | 650 | 650 |
| testes | 13,711 | 81.9 | 11,129 | 80.2 | 150 | 150 |
| watsc | 12,865 | 83.1 | 10,564 | 84.6 | 650 | 650 |

These values come from the refs4 transfer (Stage 12, job **12047419**, target sum 6,400); "Embedded" excludes pseudo-cells under the 200-gene minimum (38 LUNG goblet pseudo-cells in human space). Count mass is the fraction of BayesPrism posterior expression retained after ortholog projection. Human-space embeddings use the same v2 model as Stage 9 with species identifier 0.

How far does re-expression as human move each cell? The cosine between a pseudo-cell's rat-space and human-space embedding has a median over tissues of 0.969. This does **not** mean the same cell lands near its rat embedding: two different pseudo-cells of the same tissue are already closer in rat space than a cell is to its own human-space embedding. The cosine mostly measures how far apart the species' regions of the embedding sit. Whether the exercise signal survives is the comparison below. Whether the transferred cells land on their real-human counterparts is tested separately against Tabula Sapiens (`translation/ATLAS_PROJECTION_RESULTS.md`: PASS, macro accuracy 0.404 vs 0.084 by chance).


In [2]:
import numpy as np, matplotlib.pyplot as plt, pathlib

# Per-tissue transfer stats (from the v2 Stage 12 log, job 12014308; n_cells from tokenize_summary.json)
TRANSFER = {
    "bat":    {"rat_genes":14162,"pct_ortho":82.6,"eligible":11565,"count_mass":83.8,"n_cells":300},
    "blood":  {"rat_genes":11305,"pct_ortho":82.8,"eligible":9270, "count_mass":79.6,"n_cells":700},
    "cortex": {"rat_genes":15403,"pct_ortho":82.0,"eligible":12474,"count_mass":86.6,"n_cells":550},
    "heart":  {"rat_genes":15864,"pct_ortho":81.3,"eligible":12734,"count_mass":84.7,"n_cells":800},
    "hippoc": {"rat_genes":15523,"pct_ortho":81.2,"eligible":12419,"count_mass":86.7,"n_cells":900},
    "hypoth": {"rat_genes":14061,"pct_ortho":83.1,"eligible":11597,"count_mass":85.9,"n_cells":650},
    "kidney": {"rat_genes":14952,"pct_ortho":81.8,"eligible":12070,"count_mass":84.4,"n_cells":850},
    "liver":  {"rat_genes":12759,"pct_ortho":83.0,"eligible":10467,"count_mass":81.5,"n_cells":300},
    "lung":   {"rat_genes":15233,"pct_ortho":82.4,"eligible":12416,"count_mass":84.5,"n_cells":1400},
    "skmgn":  {"rat_genes":13144,"pct_ortho":83.3,"eligible":10843,"count_mass":86.2,"n_cells":250},
    "skmvl":  {"rat_genes":13143,"pct_ortho":83.3,"eligible":10842,"count_mass":85.4,"n_cells":250},
    "smlint": {"rat_genes":13008,"pct_ortho":82.6,"eligible":10589,"count_mass":80.5,"n_cells":700},
    "testes": {"rat_genes":13711,"pct_ortho":81.9,"eligible":11129,"count_mass":80.2,"n_cells":150},
    "watsc":  {"rat_genes":12890,"pct_ortho":83.0,"eligible":10579,"count_mass":84.6,"n_cells":650},
}
PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}
tissues = list(TRANSFER.keys())
x = np.arange(len(tissues))
w = 0.38

pct_ortho  = [TRANSFER[t]["pct_ortho"]   for t in tissues]
count_mass = [TRANSFER[t]["count_mass"]   for t in tissues]
n_cells    = [TRANSFER[t]["n_cells"]      for t in tissues]
colors     = [PALETTE[t] for t in tissues]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)

# Left: dual bar — pct_ortho + count_mass
ax = axes[0]
ax.bar(x - w/2, pct_ortho,  w, color=colors, alpha=0.9, label="Gene mapping (%)", edgecolor="k", lw=0.4)
ax.bar(x + w/2, count_mass, w, color=colors, alpha=0.55,label="Count-mass (%)",  edgecolor="k", lw=0.4, hatch="///")
ax.axhline(80, color="gray", lw=1.0, ls="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([t.upper() for t in tissues], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Percentage (%)", fontsize=10)
ax.set_title("Ortholog gene mapping per tissue\n(solid=gene%, hatched=count-mass%)", fontsize=11)
ax.set_ylim(70, 92)
ax.legend(fontsize=9)

for i, (pg, cm) in enumerate(zip(pct_ortho, count_mass)):
    ax.text(i - w/2, pg + 0.15, f"{pg:.1f}", ha="center", fontsize=7, rotation=0)
    ax.text(i + w/2, cm + 0.15, f"{cm:.1f}", ha="center", fontsize=7, rotation=0)

# Right: total pseudo-cells per tissue (horizontal bar, sorted)
ax = axes[1]
order = np.argsort(n_cells)[::-1]
t_ord = [tissues[i] for i in order]
n_ord = [n_cells[i] for i in order]
c_ord = [PALETTE[t] for t in t_ord]
ax.barh(range(len(tissues)), n_ord, color=c_ord, alpha=0.8, edgecolor="k", lw=0.4)
for j,(t,n) in enumerate(zip(t_ord, n_ord)):
    ax.text(n + 15, j, f"{n}", va="center", fontsize=8.5)
ax.set_yticks(range(len(tissues)))
ax.set_yticklabels([t.upper() for t in t_ord], fontsize=9)
ax.set_xlabel("Human-space pseudo-cells", fontsize=10)
ax.set_title(f"Pseudo-cells transferred to human space\n(total {sum(n_cells):,} across {len(tissues)} tissues)", fontsize=11)
ax.set_xlim(0, max(n_cells) * 1.14)

plt.show()
print(f"total human-space pseudo-cells: {sum(n_cells)}")
print(f"mean gene mapping: {np.mean(pct_ortho):.1f}%  count-mass: {np.mean(count_mass):.1f}%")


total human-space pseudo-cells: 8450
mean gene mapping: 82.5%  count-mass: 83.9%


## Steps 3-4: Transfer Validity and Exercise-Axis Comparison

The gate's probe is run on the human-space embedding of the same cells by the same method as the rat side (`compare_transfer.py`). Each block gets one status: PRESERVED (human-space trained AUC ≥ 0.65 with p < 0.05), WEAKENED (AUC ≥ 0.65 with p ≥ 0.05) or LOST (AUC < 0.65).

Sex is the positive control. On refs4 `compare_transfer.py` prints "transfer-validity: PASS": the human-space median sex AUC is 0.730 (rat 0.717) against its floor of 0.70, and block by block the sex AUC carries over at Spearman 0.918 across 159 blocks. (With the July references the rat-space median, 0.682, sat below the floor, so the check printed FAIL.) Exercise-axis preservation is presented as descriptive evidence.

In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

GCIH = "../data/deconvolution/genecompass_input_v2_refs4_human/gate_rcv10"
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: sex axis — rat vs human sup_sex_auc
ax = axes[0]
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    ax.scatter(sub["sup_sex_auc_rat"], sub["sup_sex_auc_human"],
               c=PALETTE.get(t,"gray"), s=22, alpha=0.6, label=t.upper())

rho, pval = spearmanr(tc["sup_sex_auc_rat"], tc["sup_sex_auc_human"], nan_policy="omit")  # TESTES: no sex AUC
lo = min(tc["sup_sex_auc_rat"].min(), tc["sup_sex_auc_human"].min()) - 0.03
hi = max(tc["sup_sex_auc_rat"].max(), tc["sup_sex_auc_human"].max()) + 0.03
ax.plot([lo,hi],[lo,hi], "k--", lw=1.0, alpha=0.4, label="Identity")
ax.axhline(0.70, color="#27AE60", lw=1.3, ls=":", alpha=0.8, label="Validity floor (0.70)")
ax.set_xlabel("Rat sex AUC (PLS-1, species=2)", fontsize=10)
ax.set_ylabel("Human sex AUC (PLS-1, species=0)", fontsize=10)
ax.set_title(f"Sex-axis positive control\nSpearman r={rho:.3f} (p={pval:.2e}, n={tc['sup_sex_auc_human'].notna().sum()})", fontsize=11)
med_r = tc["sup_sex_auc_rat"].median()
med_h = tc["sup_sex_auc_human"].median()
ax.text(0.98, 0.04, f"rat median: {med_r:.3f}\nhuman median: {med_h:.3f}\nscript check (human median >= 0.70): {'PASS' if med_h >= 0.70 else 'FAIL'}",
        transform=ax.transAxes, ha="right", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FDEDEC", edgecolor="#C0392B"))
ax.legend(fontsize=7, ncol=2, loc="upper left")

# Right: global fidelity — rat trained AUC vs human trained AUC
ax = axes[1]
hot = tc["is_hotspot"] == True
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    m   = sub["is_hotspot"] == True
    ax.scatter(sub.loc[~m,"sup_trained_auc_rat"], sub.loc[~m,"sup_trained_auc_human"],
               c=PALETTE.get(t,"gray"), s=22, alpha=0.5, label=t.upper())
    ax.scatter(sub.loc[m,"sup_trained_auc_rat"],  sub.loc[m,"sup_trained_auc_human"],
               c=PALETTE.get(t,"gray"), s=75, edgecolors="k", lw=1.2, zorder=5)

rho2, pval2 = spearmanr(tc["sup_trained_auc_rat"], tc["sup_trained_auc_human"])
lo2 = min(tc["sup_trained_auc_rat"].min(), tc["sup_trained_auc_human"].min()) - 0.02
hi2 = max(tc["sup_trained_auc_rat"].max(), tc["sup_trained_auc_human"].max()) + 0.02
ax.plot([lo2,hi2],[lo2,hi2], "k--", lw=1.0, alpha=0.4)
ax.axhline(0.65, color="#E74C3C", lw=1.2, ls=":", alpha=0.7, label="Preserved threshold (0.65)")
ax.set_xlabel("Rat trained AUC (PLS-1, species=2)", fontsize=10)
ax.set_ylabel("Human trained AUC (PLS-1, species=0)", fontsize=10)
ax.set_title(f"Global fidelity: exercise axis\nSpearman r={rho2:.3f} (p={pval2:.2e}, n={len(tc)})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.text(0.98, 0.04, "●  hotspot (rat q<0.05)\n○  other", transform=ax.transAxes,
        ha="right", fontsize=8, color="#444")

plt.show()
print(f"sex validity: rat_med={med_r:.3f} human_med={med_h:.3f} r={rho:.3f}")
print(f"exercise fidelity Spearman: r={rho2:.3f}")


sex validity: rat_med=0.717 human_med=0.730 r=0.918
exercise fidelity Spearman: r=nan


## Hotspot Survival: Descriptive Result

All **11** Stage 11 hotspots are classified PRESERVED in human space (0 weakened, 0 lost). Over all 165 blocks the counts are 42 preserved, 15 weakened, 107 lost and 1 with no human block.

- The largest drops: hypoth oligodendrocyte precursor cells, AUC 0.798 to 0.692 (delta −0.106), and skmvl vascular smooth muscle cells, 0.874 to 0.811 (−0.063); both stay above 0.65.
- The largest gains: skmgn tenocytes, 0.825 to 0.925 (+0.100), and lung endothelial cells, 0.786 to 0.841 (+0.055).

Block ordering carries over better than with the July references: human-space trained AUCs follow rat-space AUCs at Spearman 0.751 (July 0.616), and dose ρ at 0.670. These classifications are descriptive and do not establish biological conservation: no human exercise data enter the comparison.

In [4]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

GCIH = "../data/deconvolution/genecompass_input_v2_refs4_human/gate_rcv10"
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
hot  = tc[tc["is_hotspot"]==True].copy()
hot  = hot.sort_values("sup_trained_auc_rat", ascending=False).reset_index(drop=True)

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}
STATUS_COLOR = {"PRESERVED":"#27AE60","LOST":"#E74C3C","WEAKENED":"#F39C12","no_human_block":"#BDC3C7"}

fig, axes = plt.subplots(1, 2, figsize=(15, 9), constrained_layout=True)

# Left: paired AUC bars (rat vs human) per hotspot
ax = axes[0]
y  = np.arange(len(hot))
w  = 0.35
rat_auc  = hot["sup_trained_auc_rat"].values
hum_auc  = hot["sup_trained_auc_human"].values
statuses = hot["status"].values

bars_r = ax.barh(y+w/2, rat_auc,  w, color="#2980B9", alpha=0.8, label="Rat (species=2)")
bars_h = ax.barh(y-w/2, hum_auc,  w,
                 color=[STATUS_COLOR.get(s,"#95A5A6") for s in statuses],
                 alpha=0.85, label="Human (species=0, color=status)")

ax.axvline(0.65, color="gray", lw=1.2, ls=":", alpha=0.7, label="Preserved threshold (0.65)")
ax.axvline(0.5,  color="gray", lw=0.8, ls="--", alpha=0.4)
ax.set_yticks(y)
labels = [f"{r['tissue']} / {r['cell_type']}" for _,r in hot.iterrows()]
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Supervised trained AUC (PLS-1)", fontsize=10)
ax.set_title(f"Exercise hotspot survival\n{len(hot)} hotspots (rat q<0.05); sorted by rat AUC", fontsize=11)
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor="#27AE60", label=f"PRESERVED ({(statuses == 'PRESERVED').sum()}/{len(hot)})"),
    Patch(facecolor="#F39C12", label=f"WEAKENED ({(statuses == 'WEAKENED').sum()}/{len(hot)})"),
    Patch(facecolor="#E74C3C", label=f"LOST ({(statuses == 'LOST').sum()}/{len(hot)})"),
    Patch(facecolor="#2980B9", label="Rat AUC"),
]
ax.legend(handles=legend_els, fontsize=9, loc="lower right")

# Right: delta AUC waterfall
ax = axes[1]
delta_auc = hot["delta_auc"].values
colors    = [STATUS_COLOR.get(s,"#95A5A6") for s in statuses]
ax.barh(y, delta_auc, color=colors, alpha=0.85, edgecolor="k", lw=0.5)
ax.axvline(0, color="k", lw=0.8, alpha=0.6)
# status is set by the human-space AUC (>= 0.65) and p, not by the delta
ax.set_yticks(y)
ax.tick_params(axis="y", labelleft=False)
ax.set_xlabel("ΔAUC (human − rat)", fontsize=10)
ax.set_title("AUC change across species transfer\n(positive = amplified in human space)", fontsize=11)
from matplotlib.patches import Patch as P
ax.legend(handles=[
    P(facecolor="#27AE60", label="PRESERVED"),
    P(facecolor="#F39C12", label="WEAKENED"),
    P(facecolor="#E74C3C", label="LOST"),
], fontsize=9, loc="lower right")

plt.show()
print(hot["status"].value_counts().to_string())
print(f"delta_auc range: {delta_auc.min():.3f} to {delta_auc.max():.3f}")


status
PRESERVED    11
delta_auc range: -0.106 to 0.100


In [5]:
import pandas as pd
GCIH = "../data/deconvolution/genecompass_input_v2_refs4_human/gate_rcv10"
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
hot  = tc[tc["is_hotspot"]==True]
summary = hot.groupby("tissue").agg(
    n_hotspot   = ("status","size"),
    n_preserved = ("status", lambda x: (x=="PRESERVED").sum()),
    n_lost      = ("status", lambda x: (x=="LOST").sum()),
    rat_auc_max = ("sup_trained_auc_rat","max"),
    human_auc_max=("sup_trained_auc_human","max"),
).reset_index()
print("=== per-tissue hotspot preservation ===")
print(summary.to_string(index=False))
print(f"\nTOTAL hotspots: {len(hot)}, PRESERVED: {(hot['status']=='PRESERVED').sum()}, "
      f"LOST: {(hot['status']=='LOST').sum()}")
print(f"WEAKENED: {(hot['status']=='WEAKENED').sum()}")
print("Human-space Augur was not run for v2 (nothing downstream reads it); see docs/RETRAIN_V2_RESULTS.md.")


=== per-tissue hotspot preservation ===
tissue  n_hotspot  n_preserved  n_lost  rat_auc_max  human_auc_max
 blood          4            4       0        0.837          0.855
hypoth          1            1       0        0.798          0.692
  lung          3            3       0        0.817          0.841
 skmgn          1            1       0        0.825          0.925
 skmvl          2            2       0        0.887          0.863

TOTAL hotspots: 11, PRESERVED: 11, LOST: 0
WEAKENED: 0
Human-space Augur was not run for v2 (nothing downstream reads it); see docs/RETRAIN_V2_RESULTS.md.


In [6]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

GCIH = "../data/deconvolution/genecompass_input_v2_refs4_human/gate_rcv10"
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
tc["is_hotspot_bool"] = tc["is_hotspot"]==True

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, ax = plt.subplots(1, 1, figsize=(7.5, 5.5), constrained_layout=True)

# Rat vs human dose rho scatter (all 172 blocks)
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    m   = sub["is_hotspot_bool"]
    ax.scatter(sub.loc[~m,"sup_dose_rho_rat"], sub.loc[~m,"sup_dose_rho_human"],
               c=PALETTE.get(t,"gray"), s=20, alpha=0.45, label=t.upper())
    ax.scatter(sub.loc[m,"sup_dose_rho_rat"],  sub.loc[m,"sup_dose_rho_human"],
               c=PALETTE.get(t,"gray"), s=65, edgecolors="k", lw=1.1, zorder=5)

rho, pval = spearmanr(tc["sup_dose_rho_rat"], tc["sup_dose_rho_human"])
lo = min(tc["sup_dose_rho_rat"].min(), tc["sup_dose_rho_human"].min()) - 0.05
hi = max(tc["sup_dose_rho_rat"].max(), tc["sup_dose_rho_human"].max()) + 0.05
ax.plot([lo,hi],[lo,hi], "k--", lw=1.0, alpha=0.3)
ax.axhline(0, color="gray", lw=0.8, ls="--", alpha=0.4)
ax.axvline(0, color="gray", lw=0.8, ls="--", alpha=0.4)
ax.set_xlabel("Rat dose ρ (ordinal, species=2)", fontsize=10)
ax.set_ylabel("Human dose ρ (ordinal, species=0)", fontsize=10)
ax.set_title(f"Ordinal dose-response transfer\nSpearman r={rho:.3f} (p={pval:.2e}, n={len(tc)})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.text(0.98, 0.04, "●  hotspot\n○  other", transform=ax.transAxes, ha="right", fontsize=8, color="#444")

plt.show()
print(f"dose rho fidelity: r={rho:.3f}")
print("Rat-vs-human Augur panel omitted: human-space Augur was not run for v2.")

dose rho fidelity: r=nan
Rat-vs-human Augur panel omitted: human-space Augur was not run for v2.


## Stage Summary

Stage 12 compares 165 matched rat- and human-space blocks on the v2 embeddings of the refs4 pseudo-cells. All 11 rat-space hotspots are preserved in human space (11/0/0); over all blocks the counts are 42 preserved, 15 weakened and 107 lost. The sex check passes (human median 0.730, rat 0.717; block-level Spearman 0.918). Block ordering carries over at Spearman 0.751 (July references 0.616). The rat-to-human embedding cosine mostly reflects the distance between the species' regions of the embedding, not whether a cell keeps its identity; identity is tested against Tabula Sapiens instead (PASS, with tenocytes, LUNG vascular smooth muscle and BLOOD erythroid cells failing). Cross-species preservation is reported descriptively: the human side is the same cells re-expressed through the same model, not human exercise data.

## Reproducibility Notes

The refs4 transfer (re-expression as human at `target_sum=6400`, embedding with species 0, the human-space probe by the gate method and `compare_transfer.py`) ran in job **12047419** (2026-10-29). The July-reference v2 run is jobs 12014308/12014934/12014935.

```bash
python pipeline/run_stage12.py --jobs 48   # v2 + refs4 is the default config
#   steps 1-2: transfer + embed per tissue
#   step 3: human-space probe by the gate method (10 splits x 10,000 permutations) -> genecompass_input_v2_refs4_human/gate_rcv10/
#   step 4: compare_transfer.py against the rat gate (deconvolution.hotspots_table's folder) -> same folder
python pipeline/run_stage12.py --from 3 --jobs 48   # steps 3-4 only
```

Interpretation constraints:

1. Transfer is one-way from rat expression into human vocabulary; no human exercise dataset is used.
2. The hotspot delta-AUC range is −0.106 to +0.100.
3. No hotspot is weakened or lost under the pre-specified classification.
4. The sex check passes; hotspot preservation values are still descriptive.

In [7]:
import pandas as pd, hashlib, pathlib, numpy as np

GCIH = pathlib.Path(
    "../data/deconvolution/genecompass_input_v2_refs4_human"
)
GATE = GCIH / "gate_rcv10"
for fn in ["transfer_comparison.tsv","subspace_probe.tsv"]:
    p = GATE / fn
    md5 = hashlib.md5(p.read_bytes()).hexdigest()
    nrows = len(open(p).readlines()) - 1
    print(f"  {fn}: {nrows} rows | MD5={md5}")

tc = pd.read_csv(GATE / "transfer_comparison.tsv", sep="\t")
hot = tc[tc["is_hotspot"]==True]
print(f"\n  total blocks: {len(tc)}")
print(f"  hotspot blocks: {len(hot)}")
print(f"  PRESERVED: {(hot['status']=='PRESERVED').sum()}")
print(f"  WEAKENED: {(hot['status']=='WEAKENED').sum()}")
print(f"  LOST: {(hot['status']=='LOST').sum()}")
print(f"  sex validity (human med): {tc['sup_sex_auc_human'].median():.3f}")
print(f"  sex r(rat,human): {tc['sup_sex_auc_rat'].corr(tc['sup_sex_auc_human'], method='spearman'):.3f}")

tissues = ["bat","blood","cortex","heart","hippoc","hypoth","kidney","liver","lung","skmgn","skmvl","smlint","testes","watsc"]
total = 0
for t in tissues:
    emb = GCIH / t / "embeddings" / "cell_embeddings.npy"
    if emb.exists():
        arr = np.load(str(emb), mmap_mode="r")
        total += arr.shape[0]
print(f"\n  total human-space embeddings: {total} cells × 768 dims")


  transfer_comparison.tsv: 165 rows | MD5=ebdf3c40c742f2876138b8ed634a8ed1
  subspace_probe.tsv: 165 rows | MD5=d2d12f4890400a9318126edabf8c51df

  total blocks: 165
  hotspot blocks: 11
  PRESERVED: 11
  WEAKENED: 0
  LOST: 0
  sex validity (human med): 0.730
  sex r(rat,human): 0.918

  total human-space embeddings: 8062 cells × 768 dims
